# H1 — WTE excess for HalfCheetah randomised physics

For each task distribution (TD): collect `(s,a,r,s')`, embed every task, and report
for **both** `(s,a,r)` and `(s,a,s')`:

- **F** — per-task noise floor (mean pairwise distance of a task's embeddings across
  seeds), averaged over the TD's tasks.
- **d(T)** — per-seed mean pairwise distance across the TD's tasks, averaged over seeds.
- **excess = d(T) − F**, for L2 and cosine.

Then the metric-property checks (non-negativity, symmetry, identity, triangle) on all
tasks pooled across TDs. Shared logic lives in `scripts/wte_h1.py`.

Physics-varying (`L` in {0.5, 1.5, 3}): `(s,a,s')` carries the signal. Gymnasium HalfCheetah + RandomEnv-style param scaling (`h1_mujoco.py`).

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

CODE_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
for _p in (str(CODE_DIR), str(CODE_DIR / "scripts")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
from environments.h1_mujoco import HalfCheetahRandPhys
import wte_h1 as h1

N = 5000                                         # tuples / task
SEEDS = list(range(42, 47))                      # 5 data seeds
FEATURE_SETS = ["sar", "sas"]                    # (s,a,s') carries the signal here
REWARDS = ["default"]
MAX_SAMPLES = None
DATA = CODE_DIR / "data" / "cheetahdyn_h1"

L_BY_TD = {1: 0.5, 2: 1.5, 3: 3.0}             # log-scale limits (report TDs)
N_TASKS = 8

def _make_env(L):
    return HalfCheetahRandPhys(log_scale_limit=L)

# Fixed physics params per task, sampled once (seeded) so the floor measures
# only rollout noise, not a change of physics.
PARAMS = {}
for _td, _L in L_BY_TD.items():
    _env = _make_env(_L)
    for _i, _task in enumerate(_env.sample_tasks(N_TASKS, seed=_td)):
        PARAMS[f"td{_td}_t{_i}"] = _task
    _env.close()
TASK_DISTS = {_td: [f"td{_td}_t{_i}" for _i in range(N_TASKS)] for _td in L_BY_TD}
ALL_TASKS = [t for ts in TASK_DISTS.values() for t in ts]

_penv = _make_env(L_BY_TD[1]); _penv.set_task(PARAMS["td1_t0"])
_o, _ = _penv.reset()
SPEC = dict(obs_dim=int(np.ravel(_o).shape[0]),
            a_cols=int(_penv.action_space.shape[0]),
            discrete=False, n_actions=None)
_penv.close()

def _dir(reward, seed):
    return DATA / f"seed{seed}"

def collect(reward, seed):
    out = _dir(reward, seed); out.mkdir(parents=True, exist_ok=True)
    for _td, _L in L_BY_TD.items():
        env = _make_env(_L)
        for _i in range(N_TASKS):
            tid = f"td{_td}_t{_i}"; f = out / f"{tid}.npy"
            if f.exists():
                continue
            env.set_task(PARAMS[tid])
            obs, _ = env.reset(seed=1000 * seed + _i)
            rows = []
            while len(rows) < N:
                a = env.action_space.sample()
                nobs, r, term, trunc, _ = env.step(a)
                rows.append(np.concatenate([np.ravel(obs), np.ravel(a), [r], np.ravel(nobs)]))
                obs = nobs
                if term or trunc:
                    obs, _ = env.reset()
            np.save(f, np.asarray(rows[:N], dtype=np.float64))
        env.close()

def load_fn_for(reward):
    def load(task, seed):
        return np.load(_dir(reward, seed) / f"{task}.npy")
    return load

In [ ]:
# Collect (s,a,r,s') for every task over all seeds. Safe to re-run: skips files.
for _reward in REWARDS:
    for _seed in SEEDS:
        collect(_reward, _seed)
print("collection done:", {r: len(SEEDS) for r in REWARDS}, "seeds x",
      sum(len(v) for v in TASK_DISTS.values()), "tasks")

In [ ]:
# Embed, then per TD: F (mean per-task floor), d(T) (mean pairwise over seeds),
# and excess = d(T) - F, for both L2 and cosine, for (s,a,r) and (s,a,s').
EMB, _tables = {}, []
for _reward in REWARDS:
    _load = load_fn_for(_reward)
    _mu, _sd = h1.pooled_reward_stats(_load, ALL_TASKS, SEEDS,
                                      SPEC["obs_dim"], SPEC["a_cols"])
    for _fs in FEATURE_SETS:
        _emb = h1.compute_embeddings(
            TASK_DISTS, SEEDS, _load, SPEC["obs_dim"], SPEC["a_cols"], _fs, _mu, _sd,
            discrete=SPEC["discrete"], n_actions=SPEC["n_actions"],
            max_samples=MAX_SAMPLES)
        EMB[(_reward, _fs)] = _emb
        _t = h1.summarize(TASK_DISTS, SEEDS, _emb)
        _t.insert(0, "features", _fs)
        _t.insert(0, "reward", _reward)
        _tables.append(_t)
results = pd.concat(_tables, ignore_index=True)
results.round(4)

In [ ]:
# Metric-property checks on all tasks pooled across TDs (pairwise matrix averaged
# over seeds; identity threshold delta = mean same-task-across-seeds distance).
_cls = [h1.classify(TASK_DISTS, SEEDS, _emb).assign(reward=_r, features=_f)
        for (_r, _f), _emb in EMB.items()]
classification = pd.concat(_cls, ignore_index=True)[[
    "reward", "features", "metric", "delta", "nonneg", "symmetric",
    "identity_ok", "n_collapsed", "n_pairs", "triangle_ok", "triangle_bad", "class"]]
classification.round(4)